# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_object = dataset.metadata

print(f"{metadata_object.name}: {metadata_object.description}")
print(f"Published: {metadata_object.date_published} | License: {metadata_object.license}")
print(f"Dataset identifier: {metadata_object.identifier}")

## 2. Data Overview
Explore available record sets (tables), their fields (columns), and their `@id` identifiers.

In [ ]:
# List record sets and their fields by @id
record_sets = list(dataset.record_sets)
print("Available record sets and fields:")
recordset_info = {}
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', None)}")
    print("  Fields/Columns:")
    field_ids = []
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for f in fields:
            fid = f['@id'] if isinstance(f, dict) and '@id' in f else f
            print(f"    - Field @id: {fid}")
            field_ids.append(fid)
    recordset_info[rs['@id']] = field_ids

if not record_sets:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We reference all entities using their `@id` for reliability.

In [ ]:
# Choose the first record set @id for extraction
if record_sets:
    record_set_ids = [rs['@id'] for rs in record_sets]
    main_record_set_id = record_set_ids[0]

    # Load records into pandas DataFrame
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)

    print(f"Loaded DataFrame columns for RecordSet {main_record_set_id}:")
    print(df.columns.tolist())
    display(df.head())
else:
    print("No record sets available to extract data.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering on a numeric field, normalization, and grouping. All field/column references are handled by `@id` (as used in the previous cell).

In [ ]:
# Identify numeric columns for exploration
import numpy as np

if record_sets:
    # Try to detect numeric fields
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Possible numeric fields in RecordSet {main_record_set_id}:", numeric_candidates)
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0] # pick the first numeric column
        print(f"Selected numeric field: {numeric_field_id}")

        # Filter for values above a threshold
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean value):")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt grouping by a categorical field if available
        categorical_candidates = df.select_dtypes(include=['object']).columns.tolist()
        group_field_id = None
        for col in categorical_candidates:
            if col != numeric_field_id and df[col].nunique() < 10:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical grouping field found.")
    else:
        print("No numeric fields found to perform EDA.")
else:
    print("No data to analyze.")

## 5. Visualization
Visualize a distribution of the selected numeric field, and a relationship if categorical field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_candidates:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if group_field_id exists
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load metadata and data records from a complex FAIR² dataset using the `mlcroissant` library, explored available tables and fields by `@id`, extracted up-to-date records, and performed exploratory analysis and visualizations using only stable identifiers. This workflow can be extended to other record sets and fields for advanced scientific investigation.

**Key steps covered:**
- Reliable, identifier-based data access and exploration
- Filtering and normalization of numerical variables
- Optionally grouping and visualizing by categorical variables

Further steps could include advanced filtering, feature engineering, or machine learning analyses specific to proteomics/EV data.